In [ ]:
#!pip install -U transformers
#!pip install --upgrade gradio
#!pip install "huggingface_hub<1.0.0"
#!pip install huggingface_hub transformers

In [2]:
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import get_json_schema
import json

In [3]:
model_name = "HuggingFaceTB/SmolLM3-3B"
device='cpu' 
# allowed entries Expected one of cpu, cuda (NVIDA), ipu, xpu, mkldnn, opengl(any graphcis card should use), opencl(parallel to cuda runs on anything that is amd (cuda) more generic), ideep, hip, ve, fpga, maia, xla, lazy, vulkan (like opengl - another graphic card language), mps, meta, hpu, mtia, privateuseone device type at start of device string: gpu

#With device='opengl' get error -- RuntimeError: PyTorch is not linked with support for opengl devices
#With device='opencl' get error -- RuntimeError: PyTorch is not linked with support for opencl devices
#With device='vulkan' get error -- RuntimeError: PyTorch is not linked with support for vulkan devices

#This is not a loading with Pytorch issue this is a pytorch can't run those issues according to ai overlord G. Mini at google need to use the ones specfiic to the chip set.

# Mac uses mps HAVE on list -- but keeps restarting the kernel so won't run entire process (my guess is just something something the handling doesn't work in jupiter notebooks). 
# AMD/Intel (Windows) uses Use ROCm (Linux) or DirectML (Windows/WSL).--- which isn't here ugh
# For NVIDIA GPUs: Use CUDA, which is the primary and most optimized backend for PyTorch. HAVE on list
# For Intel GPUs: Use the XPU backend via the Intel Extension for PyTorch (IPEX). HAVE on list

tokenizer = AutoTokenizer.from_pretrained(model_name)


In [4]:
path="/Volumes/data/24.chdi.01-PHASE2/Kathryn_Connectomics/ZQ175_20260923-Female/Scalar_and_Volume/anovan_1/Genotype_AgeofTermination/Non_Erode/Subject_Data_Table.csv"
subject_data = pd.read_csv(path,delimiter='\t',low_memory=False,header=1) 
subject_data = subject_data.drop([0, 1]).reset_index(drop=True)

path="/Volumes/data/24.chdi.01-PHASE2/Kathryn_Connectomics/ZQ175_20260923-Female/Scalar_and_Volume/anovan_1/Genotype_AgeofTermination/Non_Erode/Bilateral_2/Group_Data_Table_Genotype.csv"
group_data = pd.read_csv(path,delimiter='\t',low_memory=False,header=1) 
group_data = group_data.drop([0, 1]).reset_index(drop=True)

path="/Volumes/data/24.chdi.01-PHASE2/Kathryn_Connectomics/ZQ175_20260923-Female/Scalar_and_Volume/anovan_1/Genotype_AgeofTermination/Non_Erode/Bilateral_2/Group_Statistical_Results_Genotype.csv"
group_stats = pd.read_csv(path,delimiter='\t',low_memory=False,header=0) 
group_stats = group_stats.drop([0, 1]).reset_index(drop=True)

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

In [5]:
def multiply(a: float, b: float):
    """
    A function that multiplies two numbers

    Args:
        a: The first number to multiply
        b: The second number to multiply
    """
    return a * b

def divide (a: float, b: float):
    """
    A function that divides two numbers

    Args:
        a: The first number to divide
        b: The second number to divide
    """
    return a / b

available_tools = {
    "multiply": multiply,
    "divide": divide
}

In [27]:
prompt=input("Input Your Prompt:")

Input Your Prompt: I'm trying to figure out about a statistical dataset. I need to figure out the total number of cells in the data sheet. The dataset has 5 rows and 15 columns.


In [35]:
#"What is 5 times 8."
messages_think = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages_think,
    tools=available_tools.values(),
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
generated_ids = model.generate(**model_inputs, max_new_tokens=32768)

In [45]:
import re

output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
out=tokenizer.decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

pattern = r"(</?think>|</?tool_call>)"
split_out = re.split(pattern, out)

#out=out.splitlines()
tool_vector=[]
begin_tool_call=0

for line in split_out:
    if line == '<tool_call>':
        begin_tool_call=1
        tool_vector.append(0)
    elif line == '</tool_call>':
        begin_tool_call=0
        tool_vector.append(0)
    elif begin_tool_call==1 and (line != '<tool_call>' or line != '</tool_call>'):
        tool_vector.append(1)
        my_dict = json.loads(line)   
        tool_call_name = my_dict.get("name")
        tool_call_args= my_dict.get("arguments")
        if tool_call_name in available_tools:
            func_to_call = available_tools[tool_call_name]
            # Unpack the dictionary directly into the function parameters
            result = func_to_call(**tool_call_args)
    else:
        tool_vector.append(0)
            
print(result)

75


In [46]:
print(out)

<think>I need to calculate the total number of cells in a 5x15 grid, which is simply the product of the number of rows and columns. So the total number of cells is 5 * 15 = 75. Now I will return this as a JSON object with the 'multiply' function.</think>
<tool_call>{"name": "multiply", "arguments": {"a": 5, "b": 15}}</tool_call>


In [48]:
print(tool_vector)

[0, 0, 0, 0, 0, 0, 1, 0, 0]


In [40]:
line

'<tool_call>{"name": "multiply", "arguments": {"a": 5, "b": 15}}</tool_call>'

JSONDecodeError: Expecting value: line 1 column 1 (char 0)